# 02 — Data Cleaning & ETL Pipeline

**Objective:** Build a reproducible Python cleaning pipeline. Log every transformation.

**Input:** `data/raw/online_sales_dataset.csv` (never edited)

**Output:** `data/processed/cleaned_online_sales.csv`

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_PATH = '../data/raw/online_sales_dataset.csv'
PROCESSED_PATH = '../data/processed/cleaned_online_sales.csv'

## Step 1 — Load Raw Data

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Loaded {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

## Step 2 — Remove Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Removed {before - after:,} duplicate rows  ({before:,} -> {after:,})')

## Step 3 — Handle Missing Values

In [ ]:
missing_before = df.isnull().sum()
print('Missing values BEFORE cleaning:')
print(missing_before[missing_before > 0])

In [ ]:
# 3a. Drop rows with missing CustomerID
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f'Dropped {before - len(df):,} rows with missing CustomerID')

In [ ]:
# 3b. Fill missing ShippingCost with median
median_ship = df['ShippingCost'].median()
filled = df['ShippingCost'].isnull().sum()
df['ShippingCost'] = df['ShippingCost'].fillna(median_ship)
print(f'Filled {filled:,} missing ShippingCost with median ({median_ship:.2f})')

In [ ]:
# 3c. Fill missing WarehouseLocation with 'Unknown'
filled = df['WarehouseLocation'].isnull().sum()
df['WarehouseLocation'] = df['WarehouseLocation'].fillna('Unknown')
print(f'Filled {filled:,} missing WarehouseLocation with Unknown')

In [ ]:
print('Missing values AFTER cleaning:')
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else 'None — all clean!')

## Step 4 — Remove Negative Quantities

In [ ]:
neg_qty = (df['Quantity'] < 0).sum()
print(f'Rows with negative Quantity: {neg_qty:,}')
before = len(df)
df = df[df['Quantity'] > 0].copy()
print(f'Removed {before - len(df):,} negative/zero quantity rows')

## Step 5 — Fix Data Inconsistencies

In [ ]:
# Fix 'paypall' typo
df['PaymentMethod'] = df['PaymentMethod'].replace({'paypall': 'PayPal'})
print('PaymentMethod values:', df['PaymentMethod'].value_counts().to_dict())

# Standardize ReturnStatus
df['ReturnStatus'] = df['ReturnStatus'].str.strip().str.title()
print('ReturnStatus values:', df['ReturnStatus'].value_counts().to_dict())

# Clip Discount to [0, 1]
df['Discount'] = df['Discount'].clip(0, 1)

## Step 6 — Convert Data Types

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)
print(df[['InvoiceDate', 'CustomerID']].dtypes)

## Step 7 — Feature Engineering

In [ ]:
df['Revenue'] = (df['Quantity'] * df['UnitPrice'] * (1 - df['Discount'])).round(2)
df['IsReturn'] = (df['ReturnStatus'] == 'Returned').astype(int)
df['OrderMonth'] = df['InvoiceDate'].dt.month
df['OrderYear'] = df['InvoiceDate'].dt.year
df['OrderDayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['OrderHour'] = df['InvoiceDate'].dt.hour

print('New columns:', ['Revenue', 'IsReturn', 'OrderMonth', 'OrderYear', 'OrderDayOfWeek', 'OrderHour'])
df[['Quantity', 'UnitPrice', 'Discount', 'Revenue', 'ReturnStatus', 'IsReturn']].head(10)

## Step 8 — Final Validation

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Rows   : {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'Missing: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')
print(f'Return rate: {df["IsReturn"].mean()*100:.2f}%')

## Step 9 — Save Cleaned Data

In [ ]:
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved to: {PROCESSED_PATH}')
print(f'File size: {os.path.getsize(PROCESSED_PATH) / 1e6:.2f} MB')